# Avian shape sweep

Avian lungs at fixed size with the shape swept: `n_parabronchi` x `l_parabronchi` = 2^k for k = 4 ... 10, so N = 2^k + 3 (19 to 1027) and `n_parabronchi` runs over every power of two (56 shapes), each undirected and directed. Each undirected shape has 100 random graphs of its own N and E.
Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.

Mean times are conditional on fixation. `fixation_prob` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
from moran_graphs.PopulationGraph import PopulationGraph

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from moran_graphs import theory
from moran_graphs.plots import COLORS, LABELS, heatmap, lines, load, print_batch, print_run_report
from moran_graphs.PopulationGraph import PopulationGraph

The six shapes at N = 35 (`n_parabronchi` x `l_parabronchi` = 32): posterior air sac left, anterior right, trachea at the bottom.
`n_parabronchi` = E - N + 1 is the number of independent cycles: n = 1 is the cycle C_N, and l = 1 joins every parabronchus straight to both air sacs, a double hub.

In [ ]:
shapes = [(2**a, 2**(5 - a)) for a in range(6)]
fig, axes = plt.subplots(1, len(shapes), figsize=(15, 2.5), layout="constrained")
for ax, (n, l) in zip(axes, shapes):
    lung = PopulationGraph.avian_graph(n_parabronchi=n, l_parabronchi=l).graph
    nx.draw(lung, nx.get_node_attributes(lung, "pos"), ax=ax, node_size=8, width=0.5, node_color=COLORS["Avian lung"])
    ax.set_title(f"n = {n}, l = {l}", fontsize=9)

## Configuration

The only cell to edit. Each figure is a list of rows of panels, `(column, title, scale)`; any column listed under Load can go in a panel.

In [ ]:
BATCHES = ["2026-10-25-avian-shape-sweep"]
R_VALUES = [1.1]  # the figures need exactly one
# any two of "n_nodes", "n_parabronchi", "l_parabronchi"; N = n x l + 3 fixes the third
X = "n_parabronchi"  # x axis of the lines, columns of the heatmaps
BY = "n_nodes"  # one line per value, rows of the heatmaps
UNDIRECTED_PANELS = [
    [("fixation_prob", LABELS["fixation_prob"], "linear"),
     ("fixation_prob_over_random", LABELS["fixation_prob_over_random"], "linear"),
     ("fixation_prob_random_percentile", "fixation probability, random percentile", "linear")],
    [("mean_fixation_time", LABELS["mean_fixation_time"], "log"),
     ("mean_fixation_time_over_random", LABELS["mean_fixation_time_over_random"], "log"),
     ("mean_fixation_time_random_percentile", "mean fixation time, random percentile", "linear")],
]
DIRECTED_PANELS = [
    [("fixation_prob", f"{LABELS['fixation_prob']}, directed", "linear"),
     ("fixation_prob_over_undirected", "fixation probability, directed / undirected", "linear"),
     ("fixation_prob_over_random", f"{LABELS['fixation_prob_over_random']}, directed", "linear")],
    [("mean_fixation_time", f"{LABELS['mean_fixation_time']}, directed", "log"),
     ("mean_fixation_time_over_undirected", "mean fixation time, directed / undirected", "log"),
     ("mean_fixation_time_over_random", f"{LABELS['mean_fixation_time_over_random']}, directed", "log")],
]

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

Three tables: `undirected` and `directed` avian shapes, and `random`. Besides the report's columns (`<metric>` is `fixation_prob` or `mean_fixation_time`):
- `n_parabronchi`, `l_parabronchi`: the shape. A random graph gets those of the avian shape it matches (E - N + 1, and (N - 3) / that), so it sits under that shape on any axis.
- `<metric>_over_random` (from `load`): over the mean of the random graphs with the same N, E and r. Directed shapes have the same N and E, but the random graphs are undirected.
- `<metric>_random_percentile`: the share of those random graphs below the shape, in %. Every random graph ran as many repeats as the shape, so it carries the same Monte Carlo noise, which makes the percentile a fair test of "just another random graph". `<metric>_stands_out` (from `load`) is True outside their middle 95% (below 2.5 or above 97.5).
- Directed only, `<metric>_over_undirected`: over the same shape undirected, with a 95% interval `_lo`/`_hi` on the log ratio (Katz for `fixation_prob`, delta method for time) and `_stands_out` where the interval excludes 1.

`_lo`/`_hi` give error bars in the line plots, `_stands_out` bold cells in the heatmaps.

In [ ]:
results, _ = load(BATCHES)
results = results[results["r"].isin(R_VALUES)]

size = ["n_nodes", "n_edges", "r"]
metrics = ["fixation_prob", "mean_fixation_time"]

random = results[results["category"] == "Random"]
random = random.assign(n_parabronchi=random["n_edges"] - random["n_nodes"] + 1,  # avian E = N - 1 + n_parabronchi
                       l_parabronchi=lambda df: (df["n_nodes"] - 3) / df["n_parabronchi"])
avian = results[results["category"] == "Avian lung"]
recipe = pd.json_normalize(avian["kwargs"].map(json.loads).tolist()).set_index(avian.index)
avian = avian.assign(n_parabronchi=recipe["n_parabronchi"], l_parabronchi=recipe["l_parabronchi"])

pairs = avian[size + metrics].reset_index().merge(random[size + metrics], on=size, suffixes=("", "_random"))
below = pd.DataFrame({f"{y}_random_percentile": 100 * (pairs[f"{y}_random"] < pairs[y]) for y in metrics})
avian = avian.join(below.groupby(pairs["index"]).mean())

In [ ]:
# SE of the log: Katz for a proportion, sqrt((1 - fixation_prob) / n_fixed); delta method for a mean, SE / mean
avian = avian.assign(fixation_prob_log_se=np.sqrt((1 - avian["fixation_prob"]) / avian["n_fixed"]),
                     mean_fixation_time_log_se=avian["mean_fixation_time_se"] / avian["mean_fixation_time"])
undirected, directed = avian[~avian["is_directed"]], avian[avian["is_directed"]]

shape = ["n_nodes", "n_parabronchi"]  # any two of N, n, l identify a shape
log_se = [f"{y}_log_se" for y in metrics]
directed = directed.join(undirected.set_index(shape)[metrics + log_se], on=shape, rsuffix="_undirected")
for y in metrics:
    ratio = directed[y] / directed[f"{y}_undirected"]
    spread = np.exp(1.96 * np.hypot(directed[f"{y}_log_se"], directed[f"{y}_log_se_undirected"]))
    lo, hi = ratio / spread, ratio * spread
    directed = directed.assign(**{f"{y}_over_undirected": ratio, f"{y}_over_undirected_lo": lo,
                                  f"{y}_over_undirected_hi": hi, f"{y}_over_undirected_stands_out": (lo > 1) | (hi < 1)})
print(f"{len(undirected)} undirected and {len(directed)} directed avian graphs, {len(random)} random graphs")

## Check: n = 1 is the cycle

With one parabronchus the avian graph is the cycle C_N, undirected or directed, whose fixation probability and time are exact (`theory.cycle`, Bd).
The simulation should agree within its 95% interval (Wilson for `fixation_prob`, 1.96 SE for time); that is 28 comparisons (14 rows, fixation probability and time), so one or two misses are expected by chance.

In [ ]:
cycles = avian[avian["n_parabronchi"] == 1].sort_values(["is_directed", "n_nodes"])
exact = np.array([theory.cycle(N, r) for N, r in zip(cycles["n_nodes"], cycles["r"])])
check = cycles[["name", "fixation_prob_lo", "fixation_prob", "fixation_prob_hi", "mean_fixation_time", "mean_fixation_time_se"]].assign(
    fixation_prob_exact=exact[:, 0], time_exact=exact[:, 1])
check["fixation_prob_ok"] = check["fixation_prob_exact"].between(check["fixation_prob_lo"], check["fixation_prob_hi"])
check["time_ok"] = (check["mean_fixation_time"] - check["time_exact"]).abs() <= 1.96 * check["mean_fixation_time_se"]
check.round(4)

## Shape across sizes

One line per `BY` against `X`, with `UNDIRECTED_PANELS`, all set in Configuration. With the default (x = `n_parabronchi`, one line per N) the left end of each line is the cycle, the right end the double hub.
Undirected only, because the random graphs are undirected; directed is at the end. With the default panels:

- **Left:** fixation probability and mean fixation time (given fixation).
- **Middle:** the same over the mean random graph with the same N and E. Above 1 the shape fixes more often (top) or slower (bottom) than a random graph with as many cycles. Grey here is each random graph over that same mean: the spread a shape has to leave to stand out.
- **Right:** the shape's percentile among those random graphs. The grey band is their middle 95%; a line leaving it stands out.

Error bars are 95% intervals (Wilson for `fixation_prob`, 1.96 SE for time), left only: elsewhere the reference is the random spread.

In [ ]:
n_rows, n_columns = len(UNDIRECTED_PANELS), max(map(len, UNDIRECTED_PANELS))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(7 * n_columns, 5 * n_rows), squeeze=False, layout="constrained")
for ax_row, panel_row in zip(axes, UNDIRECTED_PANELS):
    for ax, (y, title, scale) in zip(ax_row, panel_row):
        if y in random:  # columns the random graphs have too
            ax.scatter(random[X], random[y], s=8, alpha=0.5, linewidths=0, color=COLORS["Random"],
                       label="random graphs, same N and E")
        lines(ax, undirected, X, y, by=BY)
        if "_over_" in y:
            ax.axhline(1, color="0.2", lw=1)
        if y.endswith("_percentile"):
            ax.axhspan(2.5, 97.5, color=COLORS["Random"], alpha=0.3, lw=0)
        ax.set_xscale("log", base=2)
        ax.set(yscale=scale, ylabel=title)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside right center", frameon=False, fontsize=9)
fig.suptitle(f"Undirected avian shapes against random graphs of the same N and E, r = {R_VALUES}");

### Heatmaps over the same two axes

The same panels, one cell per (`BY`, `X`).
- **Values:** time is on a log color scale because it grows by orders of magnitude with N, so those panels mostly show N.
- **Over the mean random graph**, which takes N out: red above 1, blue below, symmetric around 1, linear for fixation probability and log for time (so x2 and x1/2 are equally strong). **Bold** (in the value panels too): the shape is outside the middle 95% of its random graphs. Even a shape that is just another random graph lands there about 6% of the time, so expect about 3 bold cells per panel by chance.
- **Percentiles:** 50 is a typical random graph, 0 or 100 beyond all of them.

Grey: no shape there, since `n_parabronchi` x `l_parabronchi` = N - 3 runs only from 16 to 1024.

In [ ]:
n_rows, n_columns = len(UNDIRECTED_PANELS), max(map(len, UNDIRECTED_PANELS))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(8 * n_columns, 4.5 * n_rows), squeeze=False, layout="constrained")
for ax_row, panel_row in zip(axes, UNDIRECTED_PANELS):
    for ax, (y, title, scale) in zip(ax_row, panel_row):
        heatmap(ax, undirected, X, y, BY, title, scale)
fig.suptitle(f"Undirected avian shapes over {LABELS.get(BY, BY)} and {LABELS.get(X, X)}, r = {R_VALUES}");

## Directed: one-way airflow

In the bird, air flows one way: posterior air sac, parabronchi, anterior air sac, trachea, back to the posterior sac. The directed avian graph follows that flow and stays strongly connected. It keeps every edge, pointed one way, so it has the same N and E as the undirected shape. With the default `DIRECTED_PANELS`:

- **Middle:** over the undirected graph of the same shape, the effect of the one-way flow alone. Above 1 (red) it raises fixation probability (top) or slows fixation (bottom). **Bold**: the 95% interval excludes 1. The n = 1 shapes are the cycle, where directed and undirected are equal in theory, so they show the noise level.
- **Right:** over the mean random graph with the same N and E. The random graphs are undirected, so this mixes shape and one-way flow: it equals middle x (undirected / random) from the heatmaps above. **Bold**: outside the middle 95% of the random graphs.

In [ ]:
n_rows, n_columns = len(DIRECTED_PANELS), max(map(len, DIRECTED_PANELS))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(8 * n_columns, 4.5 * n_rows), squeeze=False, layout="constrained")
for ax_row, panel_row in zip(axes, DIRECTED_PANELS):
    for ax, (y, title, scale) in zip(ax_row, panel_row):
        heatmap(ax, directed, X, y, BY, title, scale)
fig.suptitle(f"Directed avian shapes over {LABELS.get(BY, BY)} and {LABELS.get(X, X)}, r = {R_VALUES}");